# Phase F1 — Auto-label ~37k Warhammer images on Colab

**What you do (once):**
1. Upload `photoanalyzer_f1_bundle.tar` to the *root* of your Google Drive (MyDrive).
2. Runtime → Change runtime type → **T4 GPU**.
3. Runtime → Run all.
4. Walk away. The notebook is resumable — if Colab disconnects, re-run and it picks up where it left off.
5. When finished, download `f1_outputs.tar` from Drive to your repo, then `tar -xf f1_outputs.tar` (yields `data/pseudo_labels/`).

Expected runtime on T4: **~10 hours** for 37k images. Drive output is checkpointed every 200 images.

## 1. Mount Drive + verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), (
    'No GPU attached. Runtime → Change runtime type → T4 GPU, then Runtime → Run all.'
)
print('GPU:', torch.cuda.get_device_name(0))

## 2. Extract the bundle

In [ ]:
import os, subprocess, pathlib, shutil

BUNDLE = '/content/drive/MyDrive/photoanalyzer_f1_bundle.tar'
WORK = pathlib.Path('/content/photoanalyzer')
EXTRACT_MARKER = WORK / '.bundle_mtime'

assert os.path.exists(BUNDLE), (
    f'Bundle not found at {BUNDLE}. Upload it to the root of your Google Drive '
    'and re-run this cell.'
)

bundle_mtime = os.path.getmtime(BUNDLE)
prev_mtime = None
if EXTRACT_MARKER.exists():
    try:
        prev_mtime = float(EXTRACT_MARKER.read_text().strip())
    except Exception:
        prev_mtime = None

# If the bundle on Drive is newer than what we extracted last time —
# OR we've never extracted before — wipe + re-extract. This stops stale
# runtime state from masking new uploads (the bug that turned a fresh
# autolabel push into a re-run of the prior bench, 2026-04-25).
if prev_mtime is None or bundle_mtime > prev_mtime + 1:
    if WORK.exists():
        print(f'Wiping stale extract at {WORK} (prev_mtime={prev_mtime}, bundle_mtime={bundle_mtime})')
        shutil.rmtree(WORK)
    WORK.mkdir(parents=True)
    print(f'Extracting {BUNDLE} to {WORK} ...')
    subprocess.run(['tar', '-xf', BUNDLE, '-C', str(WORK)], check=True)
    EXTRACT_MARKER.write_text(str(bundle_mtime))
else:
    print(f'Bundle already extracted (mtime {bundle_mtime} matches) - skipping.')

n_images = sum(1 for _ in (WORK / 'backend' / 'training_data').rglob('*.jpg'))
n_ann = sum(1 for _ in (WORK / 'backend' / 'training_data_annotations').glob('*.json'))
has_phase_c = (WORK / 'PHASE_C_BENCH_MODE').exists()
print(f'{n_images} images, {n_ann} annotation JSONs, '
      f'PHASE_C_BENCH_MODE={"yes" if has_phase_c else "no"}.')


## 3. Install Python dependencies

In [ ]:
# F1 ensemble pipeline deps. transformers 4.49 is needed for SAM 3, SAM 2,
# and OWLv2's image_guided_detection. Drop the cu13 torch upgrade trap by
# pinning only to pure-Python deps; Colab's preinstalled cu12 torch works
# fine with these weights.
!pip install -q 'transformers>=4.49' 'huggingface-hub>=0.26' tqdm

import os, torch, transformers
print('torch', torch.__version__, '  transformers', transformers.__version__)

# HuggingFace token for SAM 3 (gated model). Reads from Colab's Secrets
# panel — left sidebar key icon → '+ Add secret' → name it
# `HUGGINGFACE_HUB_TOKEN` and paste your hf_... value. One-time setup.
# Falls back to the HF_TOKEN variable below if Secrets isn't available
# (e.g. running locally).
HF_TOKEN = ''

try:
    from google.colab import userdata
    secret = userdata.get('HUGGINGFACE_HUB_TOKEN')
    if secret:
        HF_TOKEN = secret
        print('HF token loaded from Colab Secrets.')
except Exception as e:
    # userdata.SecretNotFoundError or running outside Colab
    print(f'Secrets unavailable ({type(e).__name__}); using fallback HF_TOKEN var.')

if HF_TOKEN:
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF token set — SAM 3 will be attempted.')
else:
    print('No HF token — ensemble will run WITHOUT SAM 3 (DINO-X + OWLv2 + SAM 2 refine).')


## 4. Restore prior outputs (resumable across disconnects)

If a prior Colab session got partway through, its `f1_outputs.tar` is in Drive. Pulling it back in means this session skips every image it already labelled.

In [ ]:
import os, subprocess

OUT_BUNDLE = '/content/drive/MyDrive/f1_outputs.tar'
BOXES_DIR = '/content/photoanalyzer/data/pseudo_labels/boxes'
os.makedirs(BOXES_DIR, exist_ok=True)

# In Phase C bench mode we want fresh detector predictions on every
# image — do NOT restore prior outputs (they'd just get tarred back up
# unchanged, telling us nothing about the new ensemble's quality).
if os.path.exists('/content/photoanalyzer/PHASE_C_BENCH_MODE'):
    print('Phase C bench mode — skipping prior-output restore.')
elif os.path.exists(OUT_BUNDLE):
    print(f'Restoring previous outputs from {OUT_BUNDLE} ...')
    subprocess.run(
        ['tar', '-xf', OUT_BUNDLE, '-C', '/content/photoanalyzer'],
        check=True,
    )
    n = len(os.listdir(BOXES_DIR))
    print(f'{n} previously-labelled images restored. Runner will skip them.')
else:
    print('No prior outputs - fresh run.')


## 5. Run the auto-labeler

`--shuffle` makes early progress span all sources (dakkadakka / reddit / cmon / ebay / isolation), so a mid-run check shows you diverse coverage instead of just whichever faction `rglob` hits first.

In [ ]:
%cd /content/photoanalyzer
import sys, os
sys.path.insert(0, '/content/photoanalyzer/src')
os.environ['PYTHONPATH'] = '/content/photoanalyzer/src:' + os.environ.get('PYTHONPATH', '')

# Phase F1 production architecture (settled 2026-04-25 after Phase C bench):
#   SAM 3 alone is 3x mAP@50 vs DINO-X / OWLv2 visual on dense scenes.
#   The only ensemble component that pulled its weight was SAM 2 mask
#   refinement — kept as `sam3_refined`. DINO-X / OWLv2 dropped.
#   Saves ~6x GPU time vs the full ensemble with no measured quality loss.
if os.path.exists('PHASE_C_BENCH_MODE'):
    print('Phase C bench mode — measuring SAM 3 raw vs SAM 3 + SAM 2 refine.')
    if not os.environ.get('HUGGINGFACE_HUB_TOKEN'):
        raise RuntimeError(
            'HUGGINGFACE_HUB_TOKEN missing — SAM 3 needs it. Set the secret in '
            "Colab's left sidebar (key icon -> '+ Add secret') with name "
            'HUGGINGFACE_HUB_TOKEN before retrying.'
        )
    DETECTORS = ['sam3', 'sam3_refined']
    cmd = f"PYTHONPATH=/content/photoanalyzer/src python scripts/phaseF/bench_ensemble.py --detectors {' '.join(DETECTORS)}"
    print('Running:', cmd)
    !{cmd}
else:
    FLAGS = ['--shuffle']
    if not os.environ.get('HUGGINGFACE_HUB_TOKEN'):
        FLAGS.append('--no-sam3')
    print('Auto-label mode. Running:', 'autolabel_ensemble.py', *FLAGS)
    !PYTHONPATH=/content/photoanalyzer/src python scripts/phaseF/autolabel_ensemble.py {' '.join(FLAGS)}


## 6. Save outputs to Drive

Writes `f1_outputs.tar` to the Drive root. If Colab disconnects mid-run, step 4 will restore from this bundle on the next session.

In [ ]:
import subprocess, os, glob

print('Tarballing outputs …')
# Include pseudo-labels AND any bench markdown reports produced this session.
include = []
if os.path.isdir('/content/photoanalyzer/data/pseudo_labels'):
    include.append('data/pseudo_labels')
reports = glob.glob('/content/photoanalyzer/docs/benchmarks/*.md')
for r in reports:
    include.append(os.path.relpath(r, '/content/photoanalyzer'))

if not include:
    print('Nothing to save.')
else:
    cmd = ['tar', '-cf', '/content/drive/MyDrive/f1_outputs.tar',
           '-C', '/content/photoanalyzer'] + include
    print('  ', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    print(f'  Wrote f1_outputs.tar with: {include}')


## (Optional) 7. Periodic checkpoint while still running

Only useful if you're babysitting. Runs cell 6 on a timer so the Drive bundle stays fresh in case the runtime dies. Skip if you hit Run All and walked away — cell 6 still runs at the end.

In [ ]:
# Uncomment to enable periodic checkpointing alongside the runner.
# import threading, subprocess, time
# def checkpoint_loop(interval_min=30):
#     while True:
#         time.sleep(interval_min * 60)
#         try:
#             subprocess.run([
#                 'tar', '-cf', '/content/drive/MyDrive/f1_outputs.tar',
#                 '-C', '/content/photoanalyzer', 'data/pseudo_labels',
#             ], check=True)
#             print(f'[checkpoint {time.strftime("%H:%M")}] outputs mirrored to Drive')
#         except Exception as e:
#             print(f'[checkpoint err] {e}')
# threading.Thread(target=checkpoint_loop, daemon=True).start()